In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from tqdm import tqdm
import json
import csv

torch.random.manual_seed(0)

model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct", 
    device_map="cuda", 
    torch_dtype="auto", 
    trust_remote_code=True, 
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [5]:
def construct_prompt(json_obj):
    prompt = f"""<|user|>
{json_obj['context']}
{json_obj['question']}
(a) {json_obj['ans0']}
(b) {json_obj['ans1']}
(c) {json_obj['ans2']}
Answer:<|end|>
<|assistant|>""".strip()
    return prompt

def generate_output(prompt):
    generation_args = {
    "max_new_tokens": 100,
    "return_full_text": False,
    "temperature": 0.0,
    "do_sample": False,
    }
    
    output = pipe(prompt, **generation_args)
    return output[0]['generated_text']

In [ ]:
bbq_path = 'Datasets/BBQ-Dataset/Gender_identity.jsonl'
result_path = 'Datasets/BBQ-Output/Gender-identity_Phi3.csv'

# Write the header first
with open(result_path, mode='w', newline='') as csv_file:
   writer = csv.writer(csv_file, delimiter='|')
   writer.writerow(['example_id', 'response'])

# Read jsonl file
with open(bbq_path, mode='r') as file:
    file_lines = file.readlines()
    for line in tqdm(file_lines, total=len(file_lines), unit='lines'):
        obj = json.loads(line)
        
        messages = construct_prompt(obj)
        response = generate_output(messages)
        
       # Replace newlines with \n to fit all in 1 line
        response = response.replace('\n', '\\n')
        with open(result_path, mode='a', newline='') as csv_file:
            writer = csv.writer(csv_file, delimiter='|')
            writer.writerow([obj['example_id'], response])

  0%|          | 7/5672 [00:10<2:39:46,  1.69s/lines]